#### SQL CODING

In [0]:
%sql
CREATE TABLE IF NOT EXISTS population_incremental(
  country VARCHAR(50) PRIMARY KEY,
  total_population INT
);

/* This is simple inserting logic which will add duplicate records into table */
INSERT INTO population_incremental (country, total_population)
VALUES 
  ('INDIA', 331002651), 
  ('AFRICA', 99999999),
  ('CHINA', 55555555),
  ('USA', 789456123);

/* This is upsert logic to avoid duplicate entries in table */
MERGE INTO population as target USING (SELECT DISTINCT country, total_population FROM population_incremental) as source
on source.country = target.country
WHEN MATCHED THEN
  UPDATE SET target.total_population = source.total_population
WHEN NOT MATCHED THEN
  INSERT (country, total_population) 
  VALUES (source.country, source.total_population);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,4,0,0


In [0]:
%sql
SELECT country FROM population;
-- DROP TABLE population;

country
CHINA
AFRICA
INDIA


In [0]:
%sql
UPDATE population
SET total_population = 123456789
WHERE country = "INDIA";

num_affected_rows
1


In [0]:
%sql
select * from population;
select lower(country), round(cast(total_population as real) /10000, 2) from population;
select * from population order by total_population desc, country asc;
select * from population order by total_population asc limit 2;
select * from population where country != 'INDIA';
select * from population where country like 'I%';
update population set total_population = 000000 where country = "USA";
select * from population;

country,total_population
CHINA,55555555
AFRICA,99999999
INDIA,331002651
USA,0


#### PYSPARK CODING

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("country", StringType(), True),
    StructField("total_population", IntegerType(), True)
])

data = [("USA", 1000), ("INDIA", 9999), ("IRAN", 456789), ("INDIA", 10000), ("USA", 1000)]
df = spark.createDataFrame(data, schema=schema)

df.write.mode("overwrite").saveAsTable("test.test.population")

In [0]:
from pyspark.sql.functions import col, lit, when, lower, round, desc
df.display()
df.select(col("country")).display()
df.filter(col("country")== "INDIA").display()
df.withColumn("total_population", when(col("country")=="INDIA", 123456).otherwise(col("total_population"))).display()
df.filter(col("country")!= "INDIA").display()  # In pyspark we can't delete any record as its immutable, so we can filter it

df.select(
    lower(col("country")).alias("lower_country"),
    round(col("total_population").cast("double") / 10000, 2).alias("new_population")
).display()

df.select((col("country")), col("total_population")).distinct().display()
df.select(col("country"), col("total_population")).orderBy(col("total_population").desc(), col("country").asc()).display()
df.select(col("country"), col("total_population")).orderBy(col("total_population").desc()).limit(1).display()

df_india = df.filter(col("country") == "INDIA").select(col("total_population")).first()[0]
df.filter(col("total_population") > df_india).display()

df.filter(col("country").like('I%')).display()
df.filter(col("total_population").between(100, 2000000000)).display()
df.filter(col("country").isin("INDIA", "China")).display()

df.filter(
    (col("country").isin("INDIA", "USA")) &
    (col("total_population") > 1000) &
    (col("total_population") < 500000)
).display()

df.filter(
    (col("country").isin("INDIA", "USA")) |
    (col("country") == "China")
).display()

df.filter(~col("country").isin("INDIA")).display()
df.select(lower(col("country"))).display()


country,total_population
USA,1000
INDIA,9999
IRAN,456789
INDIA,10000
USA,1000


country
USA
INDIA
IRAN
INDIA
USA


country,total_population
INDIA,9999
INDIA,10000


country,total_population
USA,1000
INDIA,123456
IRAN,456789
INDIA,123456
USA,1000


country,total_population
USA,1000
IRAN,456789
USA,1000


...

country,total_population
INDIA,9999
INDIA,10000


country,total_population
INDIA,9999
INDIA,10000


country,total_population
USA,1000
INDIA,9999
INDIA,10000
USA,1000


country,total_population
USA,1000
IRAN,456789
USA,1000


lower(country)
usa
india
iran
india
usa


#### SQL functions VS Pyspark coding

##### ---- UPPER LOWER CONCAT -----

In [0]:
%sql
select lower(country) from population;
select 'Country:'|| country || '-' || 'population:'|| total_population as new from population;

new
Country:USA-population:0
Country:CHINA-population:55555555
Country:AFRICA-population:99999999
Country:INDIA-population:331002651


In [0]:
from pyspark.sql.functions import col, lit, when, lower, round, desc, concat

df.select(lower(col("country"))).display()
df.select(concat(lit("Country: "), col("country"), lit(" - "), lit("population: "), col("total_population")).alias("Country")).display()

lower(country)
usa
india
iran
india
usa


Country
Country: USA - population: 1000
Country: INDIA - population: 9999
Country: IRAN - population: 456789
Country: INDIA - population: 10000
Country: USA - population: 1000


##### ---- REPLACE TRIM LTRIM RTRIM -----

In [0]:
# REPLACE --> its used to replace any string with another string 
### syntax : REPLACE("full string to be considered", "string to be replaced", "string to be replaced with")

# TRIM --> its used to remove any leading and trailing spaces from the string
### syntax : TRIM("string to be considered")

# LTRIM --> its used to remove any leading spaces from the string
### syntax : LTRIM("string to be considered")

# RTRIM --> its used to remove any trailing spaces from the string
### syntax : RTRIM("string to be considered")

In [0]:
%sql
select REPLACE ('This is SQL fundamentals', 'SQL', 'PYTHON');
select replace(country, 'USTTA', 'USA') from population;

select
  case
    when country = 'USA' then 'USAAAA'
    else country
  end as country
from population;

country
USAAAA
CHINA
AFRICA
INDIA


In [0]:
%sql
select '           RUPAM         ' as output;
select trim('           RUPAM           ') as output;

output
RUPAM


In [0]:
from pyspark.sql.functions import regexp_replace

df.withColumn(
    "country", 
    when(col("country") == "USA", "USAAAAAA")
    .otherwise(col("country"))
    ).display()

# same using regexp_replace
df.withColumn(
    "country",
    regexp_replace(col("country"), "USAAAAAA", "USA")
).display()

country,total_population
USAAAAAA,1000
INDIA,9999
IRAN,456789
INDIA,10000
USAAAAAA,1000


country,total_population
USA,1000
INDIA,9999
IRAN,456789
INDIA,10000
USA,1000


##### ---- SUBSTR INSTR LENGTH  -----

In [0]:
# SUBSTR --> its used to extract a substring from a string
### syntax : SUBSTR("string to be considered", "starting index", "length of substring")

# INSTR --> its used to find the index of a substring in a string
### syntax : INSTR("string to be considered", "substring to be searched")

# LENGTH --> its used to find the length of a string
### syntax : LENGTH("string to be considered")

In [0]:
%sql
select * from population;
select *, 
  substr(country, 1, 2) as country_substr, 
  instr(country, "A") as country_instr, 
  length(country) as country_len 
from population;


country,total_population,country_substr,country_instr,country_len
USA,0,US,3,3
CHINA,55555555,CH,5,5
AFRICA,99999999,AF,1,6
INDIA,331002651,IN,5,5


In [0]:
from pyspark.sql.functions import substring, instr, length

df.withColumn("country_substr", substring(col("country"), 1, 2)) \
    .withColumn("country_instr", instr(col("country"), "A")) \
    .withColumn("country_len", length(col("country"))) \
    .display()

country,total_population,country_substr,country_instr,country_len
USA,1000,US,3,3
INDIA,9999,IN,5,5
IRAN,456789,IR,3,4
INDIA,10000,IN,5,5
USA,1000,US,3,3
